In [1]:
import pandas as pd
import numpy as np
from sklearn import preprocessing

In [2]:
df = pd.read_csv('framingham_heart_disease.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'framingham_heart_disease.csv'

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.shape

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
df['TenYearCHD'].value_counts()

In [ ]:
df.isnull().any()

In [ ]:
df.isnull().sum()

In [ ]:
df['heartRate'].mean()

In [ ]:
df['education'].max()

In [ ]:
df['cigsPerDay'].mean()

In [ ]:
df['BPMeds'].max()

In [ ]:
df['totChol'].mean()

In [ ]:
df['BMI'].mean()

In [ ]:
df['glucose'].mean()

In [ ]:
df['education'].fillna(df['education'].max(),inplace=True)
df['cigsPerDay'].fillna(df['cigsPerDay'].mean(),inplace=True)
df['BPMeds'].fillna(df['BPMeds'].max(),inplace=True)
df['totChol'].fillna(df['totChol'].mean(),inplace=True)
df['BMI'].fillna(df['BMI'].mean(),inplace=True)
df['heartRate'].fillna(df['heartRate'].mean(),inplace=True)
df['glucose'].fillna(df['glucose'].mean(),inplace=True)

In [ ]:
df.isnull().sum().max()

In [ ]:
X = df.iloc[:,:15].values
X

In [ ]:
y = df.iloc[:,15].values
y

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

In [ ]:
y_test.shape

### KNN

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn import metrics
import matplotlib.pyplot as plt

#### Best value for K

In [ ]:
Ks = 50
mean_acc = np.zeros((Ks-1))
std_acc = np.zeros((Ks-1))
ConfustionMx = [];
for n in range(1,Ks):
    
    #Train Model and Predict  
    neigh = KNeighborsClassifier(n_neighbors = n).fit(x_train,y_train)
    yhat=neigh.predict(x_test)
    mean_acc[n-1] = metrics.accuracy_score(y_test, yhat)

    
    std_acc[n-1]=np.std(yhat==y_test)/np.sqrt(yhat.shape[0])

mean_acc

In [ ]:
plt.plot(range(1,Ks),mean_acc,'b')
#plt.fill_between(range(1,Ks),mean_acc - 1 * std_acc,mean_acc + 1 * std_acc, alpha=0.10)
plt.legend(('Accuracy ', '+/- 3xstd'))
plt.ylabel('Accuracy ')
plt.xlabel('Number of Nabors (K)')
plt.tight_layout()
plt.show()

In [ ]:
print( "The best accuracy was with", mean_acc.max(), "with k=", mean_acc.argmax()+1) 

### After Finding the best value of K, we train the model with that K

In [ ]:
k = 15
ModelKNN = KNeighborsClassifier(n_neighbors = k).fit(x_train,y_train)
Pred_KNN = ModelKNN.predict(x_test)

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import jaccard_similarity_score

print("Avg F1-score: %.4f" % f1_score(y_test, Pred_KNN, average='weighted', labels=np.unique(Pred_KNN)))
print("Jaccard score: %.4f" % jaccard_similarity_score(y_test, Pred_KNN))

### SVM

In [ ]:
from sklearn.svm import SVC

In [ ]:
ModelSVM = SVC(kernel='rbf',gamma='auto').fit(x_train, y_train)
Pred_SVM = ModelSVM.predict(x_test) 

print("Avg F1-score: %.4f" % f1_score(y_test, Pred_SVM, average='weighted', labels=np.unique(Pred_SVM)))
print("Jaccard score: %.4f" % jaccard_similarity_score(y_test, Pred_SVM))

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import itertools

In [ ]:
def plot_confusion_matrix(cm, classes,
                          normalize=False,
                          title='Confusion matrix',
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    print(cm)

    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(classes))
    plt.xticks(tick_marks, classes, rotation=45)
    plt.yticks(tick_marks, classes)

    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label')

In [ ]:
# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test, Pred_SVM, labels=[0,1])
np.set_printoptions(precision=2)

print (classification_report(y_test, Pred_SVM))

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['No-HeartDisease(0)','HeartDisease(1)'], normalize= False,  title='Confusion matrix')

In [ ]:
y_train.sum()

### Desicion Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

In [ ]:
ModelDTree = DecisionTreeClassifier(criterion="entropy", max_depth = 10).fit(x_train, y_train)
pred_DTree = ModelDTree.predict(x_test)
print("Avg F1-score: %.4f" % f1_score(y_test, pred_DTree, average='weighted', labels=np.unique(pred_DTree)))
print("Jaccard score: %.4f" % jaccard_similarity_score(y_test, pred_DTree))

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix

ModelLR = LogisticRegression(C=0.01, solver='liblinear').fit(x_train,y_train)
pred_LR = ModelLR.predict(x_test)

In [ ]:
print("Avg F1-score: %.4f" % f1_score(y_test, pred_LR, average='weighted', labels=np.unique(pred_LR)))
print("Jaccard score: %.4f" % jaccard_similarity_score(y_test, pred_LR))

In [ ]:
# Compute confusion matrix
cnf_matrix = confusion_matrix(y_test, pred_LR, labels=[0,1])
np.set_printoptions(precision=2)

print (classification_report(y_test, pred_LR))

# Plot non-normalized confusion matrix
plt.figure()
plot_confusion_matrix(cnf_matrix, classes=['No-HeartDisease(0)','HeartDisease(1)'],normalize= False,  title='Confusion matrix')